In [1]:
import modelling_functions as mf
import numpy as np
import pyomo.environ as pyo

Build Pyomo Modell (linearly constrained MIQP (Mixed Integer Quadratic Program))

In [2]:
model = pyo.ConcreteModel()
model.I = pyo.RangeSet(0,5)  # Index set for binary variables
model.x = pyo.Var(model.I, within=pyo.Binary)  # binary variables
model.t = pyo.Var(within=pyo.NonNegativeReals)  # real variable
model.c = pyo.Constraint(rule=lambda model: model.x[0] + model.x[1] + model.x[2] >= 1)
model.c2 = pyo.Constraint(rule=lambda model: model.x[0] + 3.5*model.x[1] + model.x[2] <= 2.25)
model.c3 = pyo.Constraint(rule=lambda model: model.x[0] + model.x[1] <= 1.5)
model.o = pyo.Objective(expr= model.x[0] - 2* model.x[3] * model.x[4] + model.x[4] - 0*model.x[3])  # objective function

Model conversion and reduction

In [3]:
inst = model.create_instance()
# turn model into linearly constraint BQP (Binary Quadratic Program)
if not np.all([variable.is_binary() for variable in [var for var in inst.component_data_objects(pyo.Var, active=True) if not var.is_fixed()]]):
    BLP = mf.binary_discretization(inst, bounds=(None, 10), precision=1, encoding='log', reduce_model=False, verbose=2)
# apply reduction to reduce number of variables and constraints
BLP_reduced, fix_variables = mf.reduce_BLP(BLP, verbose=2)
# convert to QUBO by introducing penalty terms for constraints
QUBO_model = mf.BLP_to_QUBO(BLP_reduced, penalties=2, slack_encoding='log', reduce_model=False, normalize_penalty_terms=True, verbose=2)
print('Number of variables: '+ str(len([var for var in model.component_data_objects(pyo.Var) if not var.is_fixed()])))
print('Number of constraints: ' + str(len(list(model.component_data_objects(pyo.Constraint, active=True)))))

4 binary encoding variables added
1 variables fixed
Constraint c2 can be removed when some variables are fixed
Constraint c2 deactivated
Variable x[1] fixed to 0
Constraint c3 is trivial and can be removed
Constraint c3 deactivated
Variable x[3] fixed to 1
Variable x[5] fixed to 1
Variable t_encoding_variable[0] fixed to 1
Variable t_encoding_variable[1] fixed to 1
Variable t_encoding_variable[2] fixed to 1
Variable t_encoding_variable[3] fixed to 1
Variable x[4] fixed to 1
2 trivial constraints removed
8 variables fixed
Number of variables: 7
Number of constraints: 3


Generate QUBO in array form

In [4]:
# in this form it is ready to solve with all kind of algorithms
variables, QUBO, C = mf.pyomo_QUBO_to_array(QUBO_model)
print(QUBO, C)
print(variables)

[[-1.  2.]
 [ 0. -2.]] 1
('x[0]', 'x[2]')


Check feasibility of solution arrays

In [5]:
ffilter = mf.FeasibilityFilter(BLP_reduced)  # plug in here the Pyomo model that went into mf.BLP_to_QUBO
filter = ffilter.get_filter(variables=variables) # plug in here the output variables of mf.pyomo_QUBO_to_array
true_objective = ffilter.get_true_objective_function(variables=variables) # same here. If the QUBO is an approximation, this function will return the actual objective value based on the solution vector

In [6]:
sol = [1, 1] # example solution: ['x[0]', 'x[3]']
print('Objective based on QUBO array: ', np.array(sol) @ QUBO @ np.array(sol) + C)
print('Feasibility of example solution: ', filter(sol))
print('Objective value based on variable values: ', true_objective(sol))

Objective based on QUBO array:  0.0
Feasibility of example solution:  True
Objective value based on variable values:  0.0


From here on it is some old random stuff

In [57]:
model = pyo.ConcreteModel()
model.I = pyo.RangeSet(0,5)
model.x = pyo.Var(model.I, within=pyo.Binary)
model.c = pyo.Constraint(rule=lambda model: model.x[0] + model.x[1] + model.x[2] + model.x[3] <= 2)
model.o = pyo.Objective(expr=sum(model.x[i] for i in model.I), sense=pyo.maximize)

In [ ]:
instance, objective, computation_time = mf.solve_instance(model)
print(objective)
reduced_model, fix_vars = mf.reduce_BLP(model, verbose=2)
instance, objective, computation_time = mf.solve_instance(reduced_model)
print(objective)

        0.01 seconds required to write file
        0.01 seconds required for presolve
        0.09 seconds required for solver
        0.00 seconds required to read logfile 
        0.00 seconds required to read solution file 
        0.01 seconds required for postsolve
4.0
Constraint con can be simplified
Constraint con deactivated
1 trivial constraints removed
2 constraints added
        0.00 seconds required to write file
        0.00 seconds required for presolve
        0.09 seconds required for solver
        0.00 seconds required to read logfile 
        0.00 seconds required to read solution file 
        0.01 seconds required for postsolve
4.0


In [ ]:
from modelling_functions.helpers import gcd_decimal
a, l, u = (2.4, 4.8,5), 4, 6
def rescale_to_integers(*args):
    gcd = gcd_decimal(l, u, *a)
    
    return tuple(np.array(arg) / gcd for arg in args)

a, l, u = (2.4, 4.8), 4, 6
lambda f: rescale_to_integers(a,l,u)
lambda g: clear_denominators(a, l, u)
print(timeit.timeit(f, number=1000))
print(timeit.timeit(g, number=1000))

0.2307396999995035
0.20388079999975162


In [ ]:
a, l, u = (2.4, 4.8), 4, 6
a, l, u = rescale_to_integers(a, l, u)
print(a, l, u)
a, l, u = (2.4, 4.8), 4, 6
a, l, u = clear_denominators(a, l, None)
print(a, l, u)

[ 6. 12.] 10.0 15.0
[3, 6] 5 None


In [ ]:
a, b = clear_denominators(np.pi, np.e)
a / b - np.pi / np.e

-2.353628403284347e-11

In [ ]:
import timeit
f = lambda : mf.check_trivial_constraint(model.con, verbose=0)
g = lambda : mf.check_trivial_constraint_old(model.con, verbose=0)
print(timeit.timeit(f, number=10))
print(timeit.timeit(g, number=10))

0.002749000000221713
0.002690999999686028


In [20]:
import numpy as np
import itertools
lower = 0
upper = 15
# num_bits = int(np.ceil(1 / 2 + np.sqrt(slack_upper - slack_lower - 3 / 4)))
# slack_range = range(1, num_bits + 1)
# slack_def = lambda slack: sum(i  * slack[i] for i in slack_range[:-1]) + (slack_upper - slack_lower - num_bits * (num_bits - 1) / 2) * slack[num_bits] + slack_lower

# num_bits = int(np.floor(np.log2(slack_upper - slack_lower))) + 1
# slack_range = range(num_bits)
# slack_def = lambda slack: sum(2**i * slack[i] for i in slack_range[:-1]) + (slack_upper - slack_lower - 2**(num_bits - 1) + 1) * slack[num_bits - 1] + slack_lower

coeff_bound = 2
if upper - lower < 2**(np.floor(np.log2(coeff_bound))+1):
    print('a')
    num_bits = int(np.floor(np.log2(upper - lower))) + 1
    binary_range = range(num_bits)
    encoding_def = lambda slack: sum(2**i * slack[i] for i in binary_range[:-1]) + (upper - lower - 2**(num_bits - 1) + 1) * slack[num_bits - 1] + lower
    option_a = True
elif (upper - lower >= 2**(np.floor(np.log2(coeff_bound))+1)):
    print('b')
    rho=int(np.floor(np.log2(coeff_bound)))+1
    nu=int(upper - lower-sum(2**(i-1) for i in range(1,rho+1)))
    eta=int(np.floor(nu/coeff_bound))
    binary_range=range(1, rho+eta+1+1)
    num_bits = len(binary_range)
    encoding_def = lambda slack: sum(2**(i-1) * slack[i] for i in range(1,rho+1)) + sum(coeff_bound *slack[i] for i in range(rho+1,rho+eta+1))+ (nu-eta*coeff_bound)*slack[rho+eta+1]
    encoding_penalty_def = lambda slack: 0 # no extra encoding constraints needed here
    option_a = False
else:
    print('wrong')

print(num_bits)
left = list(range(lower, upper + 1))
slacks = [{i: per[i - binary_range[0]] for i in binary_range} for per in itertools.product([0, 1], repeat=num_bits)]
for slack in slacks:
    value = encoding_def(slack)
    if not (lower <= value <= upper):
        raise Exception('Value {}'.format(value))
    if value in left:
        left.remove(value)
    print(list(slack.values()), value)
print(left)

b
9
[0, 0, 0, 0, 0, 0, 0, 0, 0] 0
[0, 0, 0, 0, 0, 0, 0, 0, 1] 0
[0, 0, 0, 0, 0, 0, 0, 1, 0] 2
[0, 0, 0, 0, 0, 0, 0, 1, 1] 2
[0, 0, 0, 0, 0, 0, 1, 0, 0] 2
[0, 0, 0, 0, 0, 0, 1, 0, 1] 2
[0, 0, 0, 0, 0, 0, 1, 1, 0] 4
[0, 0, 0, 0, 0, 0, 1, 1, 1] 4
[0, 0, 0, 0, 0, 1, 0, 0, 0] 2
[0, 0, 0, 0, 0, 1, 0, 0, 1] 2
[0, 0, 0, 0, 0, 1, 0, 1, 0] 4
[0, 0, 0, 0, 0, 1, 0, 1, 1] 4
[0, 0, 0, 0, 0, 1, 1, 0, 0] 4
[0, 0, 0, 0, 0, 1, 1, 0, 1] 4
[0, 0, 0, 0, 0, 1, 1, 1, 0] 6
[0, 0, 0, 0, 0, 1, 1, 1, 1] 6
[0, 0, 0, 0, 1, 0, 0, 0, 0] 2
[0, 0, 0, 0, 1, 0, 0, 0, 1] 2
[0, 0, 0, 0, 1, 0, 0, 1, 0] 4
[0, 0, 0, 0, 1, 0, 0, 1, 1] 4
[0, 0, 0, 0, 1, 0, 1, 0, 0] 4
[0, 0, 0, 0, 1, 0, 1, 0, 1] 4
[0, 0, 0, 0, 1, 0, 1, 1, 0] 6
[0, 0, 0, 0, 1, 0, 1, 1, 1] 6
[0, 0, 0, 0, 1, 1, 0, 0, 0] 4
[0, 0, 0, 0, 1, 1, 0, 0, 1] 4
[0, 0, 0, 0, 1, 1, 0, 1, 0] 6
[0, 0, 0, 0, 1, 1, 0, 1, 1] 6
[0, 0, 0, 0, 1, 1, 1, 0, 0] 6
[0, 0, 0, 0, 1, 1, 1, 0, 1] 6
[0, 0, 0, 0, 1, 1, 1, 1, 0] 8
[0, 0, 0, 0, 1, 1, 1, 1, 1] 8
[0, 0, 0, 1, 0, 0, 0, 0, 0] 2
[0, 0,

In [6]:
np.floor(np.log2(coeff_bound))+1

np.float64(1.0)

In [11]:
model = pyo.ConcreteModel()
model.I = pyo.RangeSet(0,5)
model.x = pyo.Var(model.I, within=pyo.PositiveReals, bounds=(0, 10.5))
model.c = pyo.Constraint(rule=lambda model: model.x[0] + model.x[1] + model.x[2] >= 1)
model.o = pyo.Objective(expr= model.x[0])

discretized_model = mf.binary_discretization(model, bounds=(None, None), precision=0.1, encoding='log', verbose=2, reversed=False)

42 binary encoding variables added
6 variables fixed


In [12]:
print(discretized_model.o.expr)

('x[0]_encoding_variable'[0] + 2*'x[0]_encoding_variable'[1] + 4*'x[0]_encoding_variable'[2] + 8*'x[0]_encoding_variable'[3] + 16*'x[0]_encoding_variable'[4] + 32*'x[0]_encoding_variable'[5] + 41*'x[0]_encoding_variable'[6] + 1)/10.0
